# 03 · 拓扑构建与基线仿真

基于 `src/network/topology.py`，验证三层供应链网络（区域仓 → 城市仓 → 终端门店）
能正确构建，并用框架默认 `OrderUpTo` 策略跑通一次完整仿真。

流程：构建拓扑 → 加载 A 的合成需求与预测 → 注入叶子门店 → 跑基线仿真 → 观察指标。

> 本 notebook 只做**拓扑与基线仿真**：不涉及协同策略（那是 04 的内容），
> 也不修改 A 的数据/预测模块。

In [10]:
# ---- 环境导入：兼容内核工作目录在项目根或 notebooks/ 两种情况 ----
import sys
from pathlib import Path

_root = Path.cwd()
if _root.name == "notebooks":
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np

from src.network.topology import build_network_from_config, run_baseline_policy
from src.data_loader import load_synthetic_demand, pivot_demand_wide
from src.forecasting.custom_forecaster import WeightedMAForecaster

print(f"项目根: {_root}")

项目根: d:\用户目录\Desktop\AI智能体校园黑客松\supply-chain-agent


## 1. 构建三层拓扑

从 `config/network_config.yaml` 读取参数，构造 `SupplyChainGraph` + `NetworkSupplyChain`。

**框架事实**（读 deepbullwhip 0.4.1 源码实测）：
- `SupplyChainGraph(nodes={名: EchelonConfig}, edges={(上游,下游): EdgeConfig})`
  —— 连接关系由 edges 字典的 **key** 表达；
- 补货提前期配在**节点**的 `EchelonConfig.lead_time` 上，`EdgeConfig.lead_time` 不生效；
- `NetworkSupplyChain(graph, policies={名: OrderingPolicy})` —— 每个节点一个独立策略实例。

In [11]:
# ---- 构建拓扑并打印结构 ----
net = build_network_from_config(_root / "config" / "network_config.yaml")

print("拓扑构建成功")
print(f"节点数: {len(net.graph.nodes)}")
print(f"边数:   {len(net.graph.edges)}")
print(f"拓扑序: {net.graph.topological_order()}")
print()
for src, dst in net.graph.edges:
    print(f"  {src:22s} -> {dst}")

拓扑构建成功
节点数: 7
边数:   6
拓扑序: ['regional_warehouse', 'city_warehouse_1', 'city_warehouse_2', 'store_1', 'store_2', 'store_3', 'store_4']

  regional_warehouse     -> city_warehouse_1
  city_warehouse_1       -> store_1
  city_warehouse_1       -> store_2
  regional_warehouse     -> city_warehouse_2
  city_warehouse_2       -> store_3
  city_warehouse_2       -> store_4


## 2. 准备需求与预测（来自数据准备模块）

- **需求**：`load_synthetic_demand()` 合成 AR(1)（4 SKU × 365 天，对齐 config 基准）；
- **映射**：取前 4 条序列依次对应 store_1..4（拓扑有 4 个门店）；
- **预测**：`WeightedMAForecaster` 逐门店输出整段 `(T,)` 的 `(fm, fs)`。

**框架接口说明 §1.4 实测**：`simulate()` 的 `forecasts_mean/std` **只对叶子门店生效**，
中间层（城市仓 / 区域仓）由框架用"最近 8 期下游订单均值"自行推算，
所以这里只需为 4 个门店准备预测。

In [12]:
# ---- 加载需求、映射门店、生成预测 ----
demand_df = load_synthetic_demand()
wide_df = pivot_demand_wide(demand_df)          # index=date, columns=SYN_SKU_1..5

store_names = ["store_1", "store_2", "store_3", "store_4"]
series_names = list(wide_df.columns)[:4]        # 取前 4 条序列
print("需求宽表形状:", wide_df.shape)
print("门店映射:", dict(zip(store_names, series_names)))

wma = WeightedMAForecaster(window=8, weights=[1, 1, 1, 1, 2, 2, 3, 4])
demand, forecasts_mean, forecasts_std = {}, {}, {}
for store, col in zip(store_names, series_names):
    series = wide_df[col].to_numpy(dtype=np.float64)
    fm, fs = wma.generate_forecasts(series)
    demand[store] = series
    forecasts_mean[store] = fm
    forecasts_std[store] = fs

print(f"\n已准备 {len(demand)} 个门店的需求与预测，序列长度 {len(demand['store_1'])}")

需求宽表形状: (365, 5)
门店映射: {'store_1': 'SYN_SKU_1', 'store_2': 'SYN_SKU_2', 'store_3': 'SYN_SKU_3', 'store_4': 'SYN_SKU_4'}

已准备 4 个门店的需求与预测，序列长度 365


## 3. 跑基线 OrderUpTo 仿真

用框架默认的 `OrderUpToPolicy` 跑一遍完整仿真，输出三个赛题指标：
`Bullwhip Ratio` / `Fill Rate` / `Total Cost`，以及各节点明细。

In [13]:
# ---- 基线仿真 ----
result = run_baseline_policy(net, demand, forecasts_mean, forecasts_std)

print(f"总成本 total_cost: {result.total_cost:,.2f}")
print(f"牛鞭系数 cumulative_bullwhip: {result.cumulative_bullwhip:.4f}")
print()
print(f"{'节点':<22s}{'订单均值':>10s}{'库存均值':>12s}{'牛鞭系数':>10s}{'成本':>12s}")
print("-" * 66)
for name, r in result.node_results.items():
    print(f"{name:<22s}{np.mean(r.orders):>10.2f}"
          f"{np.mean(r.inventory_levels):>12.2f}"
          f"{r.bullwhip_ratio:>10.2f}{r.total_cost:>12.2f}")

总成本 total_cost: 22,563.92
牛鞭系数 cumulative_bullwhip: 4.4828

节点                          订单均值        库存均值      牛鞭系数          成本
------------------------------------------------------------------
regional_warehouse         39.91      -81.60      4.48    12079.06
city_warehouse_1           19.74      -21.88      4.66     4192.39
store_1                     9.85        5.91      1.52      477.40
store_2                     9.97        6.16      1.37      474.34
city_warehouse_2           20.06      -22.59      5.55     4359.85
store_3                    10.18        6.10      1.64      475.84
store_4                     9.95        6.37      1.54      505.05


## 4. 结果观察

从基线输出可以看到两个关键现象，正是后续协同策略要改善的目标：

**现象 1：中间层持续缺货**
`regional_warehouse` 和 `city_warehouse` 的库存均值均为负（backorder 状态），
说明上游补货跟不上下游订单，整条链在"欠货"运行。

**现象 2：城市仓牛鞭放大严重**
门店的牛鞭系数约 1.4~1.6（接近需求波动本身），
但城市仓放大到 4.7~5.6，区域仓 4.5 —— 上游看到的波动被逐级放大。

**下一步（04_coordination_signal_poc）**：设计聚合缺口信号 + 协同补货策略，
目标是降低中间层的牛鞭系数与缺货，同时不显著增加总成本。

## 小结

- ✅ 三层拓扑构建正确：7 节点、6 条边、拓扑序符合预期；
- ✅ 基线 `OrderUpTo` 仿真跑通，产出三项指标与节点明细；
- ⚠️ 基线暴露出中间层缺货、牛鞭放大两个问题，作为协同策略的优化目标。

**交接物**：`NetworkSimulationResult`（node_results / edge_flows /
cumulative_bullwhip / total_cost），供 `run_benchmark.py` 与 `dashboard.py` 消费。